# Setup

In [ ]:
from eharmony.cylindrical_harmonics import CylindricalHarmonics

In [ ]:
K = 1
L = 1
M = 3
ch = CylindricalHarmonics(K, L, M, num_radii=10, num_phi=36, num_height=10)
ch.Psi.shape

# Plotting the CylindricalHarmonics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from eharmony import plotting

In [ ]:
basis_fns = ch.Psi.view(M, K, L*2+1, ch.num_radii, ch.num_phi, ch.num_height)
plotting.plot_cylinder_fn(basis_fns[0,0,0].numpy(), fig=plt.figure(figsize=(2,2)))

In [ ]:
import torch
w = torch.randn(9).view(1,9)
ch(w)[0,0,0,0]

In [ ]:
ch(w, torch.tensor([[0.05,0.0873,0.05]]))

In [ ]:
basis_fns = ch.Psi.view(M, K, L*2+1, ch.num_radii, ch.num_phi, ch.num_height)

fig = plt.figure(figsize=(4*L, M*K))
subfigs = fig.subfigures(K*M, L*2+1)
for m in range(M):
    for k in range(K):
        for l in range(L*2+1):
                plotting.plot_cylinder_fn(basis_fns[m,k,l].numpy(), fig=subfigs[m*K+k, l])

# Cylindrical Harmonics Transform

In [ ]:
import torch
from torch import nn

In [ ]:
def f(r, phi, z):
    return r**2 + np.cos(2*phi) - z**2 + r*np.sin(phi)*z

r = torch.linspace(0,1,10)
phi = torch.linspace(0,2*np.pi,36)
z = torch.linspace(0,1,10)
y = f(r[:,None,None], phi[None,:,None], z[None,None,:])

In [ ]:
plotting.plot_cylinder_fn(y.numpy(), fig=plt.figure(figsize=(2,2)))

In [ ]:
class CHT(nn.Module):
    def __init__(self, K=1, L=3, M=1, num_radii=10, num_phi=36, num_height=10):
        super().__init__()
        self.w_lin = nn.Linear(num_radii * num_phi * num_height, M*K*(L*2+1))
        self.ch = CylindricalHarmonics(K,L,M, num_radii=num_radii, num_phi=num_phi, num_height=num_height)

    def forward(self, x):
        w = self.w_lin(x)
        return self.ch(w), w.detach()

In [ ]:
K = 2
L = 3
M = 2
ch_model = CHT(K,L,M)
optimizer = torch.optim.Adam(ch_model.parameters(), lr = 1e-4)

for iter in range(200):
    y_pred, _ = ch_model(y.view(1,-1))
    loss = (y_pred - y).pow(2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if iter % 10 == 0:
        print(f'iteration: {iter} loss: {loss.item()}')

In [ ]:
with torch.no_grad():
    pred, w = ch_model(y.view(1,-1))
    plotting.plot_cylinder_fn(pred.squeeze().numpy(), fig=plt.figure(figsize=(2,2)))

In [ ]:
w_ph = torch.einsum("bwrpz,bw->bwrpz", ch_model.ch.Psi, w).view(M,K,2*L+1, ch_model.ch.num_radii, ch_model.ch.num_phi, ch_model.ch.num_height)
w.numpy().round(2)

In [ ]:
fig = plt.figure(figsize=(4*L, K*M))
subfigs = fig.subfigures(K*M, L*2+1)
for m in range(M):
    for k in range(K):
        for l in range(L*2+1):
            plotting.plot_cylinder_fn(w_ph[m,k,l].numpy(), fig=subfigs[m*k+K, l], vmin=w_ph.min(), vmax=w_ph.max())

# 